<a href="https://colab.research.google.com/github/buiduchuy28082005-lgtm/VIN-Annual-Report-NLP-Topic-Analytics-Dashboard/blob/master/NLP_VIN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1. Thu thập và xử lý dữ liệu thô

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Cài đặt
!pip -q install unidecode PyMuPDF pytesseract Pillow
!sudo apt-get -qq update
!sudo apt-get -qq install tesseract-ocr -y
!sudo apt-get -qq install tesseract-ocr-vie -y

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 24.6 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package tesseract-ocr-vie.
(Reading database ... 117528 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-vie_1%3a4.00~git30-7274cfa-1.1_a

In [ ]:
import os, re
import pandas as pd
from unidecode import unidecode
import fitz  # PyMuPDF
import pytesseract
from PIL import Image

In [ ]:
GOOD_HEADINGS_ORIGINAL = [
    "THÔNG ĐIỆP","THÔNG ĐIỆP CỦA HĐQT","THÔNG ĐIỆP CỦA TỔNG GIÁM ĐỐC","THƯ",
    "TẦM NHÌN","CHIẾN LƯỢC","ĐỊNH HƯỚNG PHÁT TRIỂN","CHIẾN LƯỢC PHÁT TRIỂN","ĐỊNH HƯỚNG","GIAI ĐOẠN",
    "TỔNG QUAN","TÌNH HÌNH","HOẠT ĐỘNG KINH DOANH","KẾT QUẢ HOẠT ĐỘNG","PHÁT TRIỂN BỀN VỮNG",
    "BÁO CÁO VÀ ĐÁNH GIÁ CỦA BAN ĐIỀU HÀNH","ĐÁNH GIÁ CỦA HỘI ĐỒNG QUẢN TRỊ","ĐÁNH GIÁ CỦA BAN QUẢN TRỊ",
    "ĐÁNH GIÁ CỦA HĐQT","ĐÁNH GIÁ CỦA BAN GIÁM ĐỐC",
    "QUẢN TRỊ RỦI RO","RỦI RO"
]

STOP_KEYWORDS = [
    "QUAN TRI CONG TY", "BAO CAO TAI CHINH", "CO CAU CO DONG", "BAO CAO KIEM TOAN", "PHU LUC",
    "THUYET MINH", "BAO CAO KIEM TOAN DOC LAP", "BAO CAO CUA BAN KIEM SOAT", "DANH SACH"
]

ALL_KEYWORDS = [unidecode(k).upper() for k in GOOD_HEADINGS_ORIGINAL] + STOP_KEYWORDS

In [ ]:
GLOBAL_DATA_FOLDER = '/content/drive/MyDrive/DT'
OUT_SEGMENTS_CSV = '/content/drive/MyDrive/DT/segments_clean.csv'

EXPORT_DOCS_YEAR = False
OUT_DOCS_YEAR_CSV = '/content/drive/MyDrive/DT/docs_year.csv'

In [ ]:
# Reset file cũ
paths_to_reset = [OUT_SEGMENTS_CSV]
if EXPORT_DOCS_YEAR and OUT_DOCS_YEAR_CSV:
    paths_to_reset.append(OUT_DOCS_YEAR_CSV)

for p in paths_to_reset:
    if os.path.exists(p):
        os.remove(p)
        print("🔄 Reset:", p)

# Chuẩn chất lượng segment
MIN_WORDS_SEGMENT = 80       # segment quá ngắn -> bỏ (giảm nhiễu)
SOFT_MAX_WORDS = 200         # mục tiêu chunk ~200 từ
HARD_MAX_WORDS = 250         # không cho segment >250 từ

# OCR config
OCR_LANG = 'vie+eng'
OCR_DPI = 200


In [ ]:
def looks_like_toc(text: str) -> bool:
    if not text:
        return False
    u = unidecode(text).upper()
    if "MUC LUC" in u:
        return True
    # dot leaders dài thường là TOC
    if re.search(r"\.{5,}", text):
        return True
    # nhiều dòng kết thúc bằng số trang
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    if len(lines) >= 10:
        end_num = sum(bool(re.search(r"\s\d{1,3}$", ln)) for ln in lines[:30])
        if end_num >= 8:
            return True
    return False

In [ ]:
CTRL_RE = re.compile(r'[\x00-\x08\x0b-\x1f]')  # control chars trừ \n\t\r

def is_garbled(text: str) -> bool:
    if not text:
        return True
    t = text.strip()
    if not t:
        return True
    ctrl = len(CTRL_RE.findall(t))
    # Một số PDF lỗi font cho ra rất nhiều ký tự control
    if ctrl >= 10:
        return True
    # Ký tự thay thế
    if "\ufffd" in t:
        return True
    return False

def needs_ocr(text: str) -> bool:
    if not text or len(text.strip()) < 80:
        return True
    if is_garbled(text):
        return True

    letters = sum(ch.isalpha() for ch in text)
    digits  = sum(ch.isdigit() for ch in text)

    # Trang quá ít chữ thật -> OCR
    if letters < 30:
        return True

    return False


def read_pdf_text_ocr_per_page(path: str, ocr_lang=OCR_LANG, ocr_dpi=OCR_DPI):
    """
    Đọc từng trang. Nếu trang có ít chữ / nghi scan -> OCR trang đó (trừ TOC).
    """
    pages_data = []
    try:
        doc = fitz.open(path)
        for i in range(len(doc)):
            page = doc[i]
            page_num = i + 1
            text_pure = page.get_text("text").strip()

            final_text = text_pure
            if needs_ocr(text_pure) and (not looks_like_toc(text_pure)):
                try:
                    pix = page.get_pixmap(dpi=ocr_dpi)
                    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
                    text_ocr = pytesseract.image_to_string(img, lang=ocr_lang).strip()
                    # chọn bản tốt hơn
                    if len(text_ocr) > max(120, len(text_pure) * 1.3):
                        final_text = text_ocr
                except:
                    pass

            pages_data.append({"page_num": page_num, "text": final_text})

        doc.close()
    except Exception as e:
        print(f" -> LỖI đọc file {os.path.basename(path)}: {e}")
        return []
    return pages_data

In [ ]:
def split_sentences_vi(text: str):
    # tách câu đơn giản cho tiếng Việt
    parts = re.split(r'(?<=[\.\!\?\…])\s+', text.strip())
    return [p.strip() for p in parts if p.strip()]

# HEADING DETECTION (dùng để lọc scope, nhưng KHÔNG xuất ra section/page)

def detect_headings(full_text: str):
    headings = []
    if not full_text:
        return headings
    for line in full_text.splitlines():
        ln = line.strip()
        if not ln:
            continue
        if len(ln.split()) < 2:
            continue

        un = unidecode(ln).upper()
        if "MUC LUC" in un or re.match(r"^CHUONG\s+[IVXLC]+", un):
            continue

        alpha = sum(1 for c in ln if c.isalpha())
        upper = sum(1 for c in ln if c.isupper())
        upper_ratio = (upper / alpha) if alpha > 0 else 0.0

        if any(k in un for k in ALL_KEYWORDS) and (upper_ratio > 0.2 or any(w.isupper() for w in ln.split()[:2])):
            headings.append(ln)

    # unique giữ thứ tự
    seen = set()
    out = []
    for h in headings:
        key = unidecode(h).upper()
        if key not in seen:
            out.append(h)
            seen.add(key)
    return out

In [ ]:
def extract_relevant_text_by_headings(pages_data):
    """
    Cắt các đoạn nằm dưới các heading thuộc GOOD_HEADINGS, bỏ các heading STOP.
    Nếu không detect được heading -> dùng toàn văn (sau đó vẫn clean mạnh).
    """
    if not pages_data:
        return ""

    full_text = "\n".join([p["text"] for p in pages_data])
    headings = detect_headings(full_text)

    if not headings:
        return full_text

    # Chỉ giữ heading GOOD, bỏ STOP
    good_keys = [unidecode(k).upper() for k in GOOD_HEADINGS_ORIGINAL]
    stop_keys = STOP_KEYWORDS

    # tìm vị trí heading
    spans = []
    start_pos = 0
    for i, h in enumerate(headings):
        idx = full_text.find(h, start_pos)
        if idx == -1:
            continue
        # skip STOP heading
        if any(sk in unidecode(h).upper() for sk in stop_keys):
            start_pos = idx + len(h)
            continue
        # chỉ giữ heading GOOD
        if not any(gk in unidecode(h).upper() for gk in good_keys):
            start_pos = idx + len(h)
            continue

        next_line_pos = full_text.find("\n", idx)
        content_start = next_line_pos + 1 if next_line_pos != -1 else idx + len(h)

        if i + 1 < len(headings):
            next_h = headings[i+1]
            content_end = full_text.find(next_h, content_start)
            if content_end == -1:
                content_end = len(full_text)
        else:
            content_end = len(full_text)

        chunk = full_text[content_start:content_end].strip()
        if len(chunk.split()) >= 30:
            spans.append(chunk)

        start_pos = content_end

    if spans:
        return "\n\n".join(spans)
    return full_text


In [ ]:
def clean_text_and_remove_tables(text: str) -> str:
    if not text:
        return ""

    # Loại bỏ email và liên kết (URL)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)

    # Loại bỏ các mẫu tiêu đề đầu trang/cuối trang (header/footer) lặp đi lặp lại
    text = re.sub(r'(?im)^\s*BCTN\s*\d{4}.*$', ' ', text)

    # Loại bỏ các từ ngữ khuôn mẫu mang tính thủ tục của "báo cáo thường niên"
    text = re.sub(r'(?i)\b(bao\s*cao\s*thuong\s*nien|annual report)\b(?:\s*(?:nam)?\s*\d{4})?', ' ', text)

    # Loại bỏ ký hiệu đánh dấu số trang
    text = re.sub(r'(?i)\b(trang|page)\s*\d+\b', ' ', text)

    # Chỉ giữ lại các chữ cái tiếng Việt + các dấu câu cơ bản
    text = re.sub(r'[^\w\u00C0-\u024F\s\.\,\;\:\!\?\…\-\/\(\)\[\]\{\}]', ' ', text)

    # Chuẩn hóa các khoảng trắng (space và tab)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'[ \u00A0]+', ' ', text)

    # Loại bỏ các dòng có cấu trúc giống dạng bảng (table)
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    kept = []
    for ln in lines:
        alpha = sum(c.isalpha() for c in ln)
        digit = sum(c.isdigit() for c in ln)

        # Nếu dòng chỉ chứa số/ký tự đặc biệt -> loại bỏ
        if re.match(r'^[\s\d\.\,\%\/\-]+$', ln):
            continue

        # Nếu dòng ngắn + chứa quá nhiều chữ số (nhiều hơn gấp đôi chữ cái) -> loại bỏ
        if len(ln.split()) < 10 and digit > alpha * 2:
            continue

        # Xử lý làm sạch nhẹ các số nằm xen kẽ trong văn bản (ví dụ: các số liệu tài chính)
        ln = re.sub(r'(\s+\d{1,3}(?:\,\d{3})*(?:\.\d{1,2})?[\s\.\,\%])', ' ', ln)
        ln = re.sub(r'\s+', ' ', ln).strip()

        # Loại bỏ dòng nếu nó chứa các từ khóa dừng (STOP_KEYWORDS)
        if any(sk.lower() in unidecode(ln).lower() for sk in STOP_KEYWORDS):
            continue

        kept.append(ln)

    text = "\n".join(kept)

    # Nối các dòng bị ngắt đơn lẻ ở trong cùng một đoạn văn
    text = re.sub(r'([^\n])\n([^\n])', r'\1 \2', text)

    # Chuẩn hóa khoảng cách ngắt giữa các đoạn văn (paragraph)
    text = re.sub(r'\n\s*\n', '\n\n', text).strip()
    text = re.sub(r'\n\n+', '\n\n', text)

    return text

In [ ]:
def segment_by_paragraph(text: str):
    if not text:
        return []
    segs = [s.strip() for s in text.split("\n\n") if s.strip()]

    segs = [s for s in segs if len(s.split()) >= 20]
    return segs

def chunk_segments(segments):
    """
    Gộp đoạn để đạt ~200 từ; nếu quá dài thì chia theo câu (tiếng Việt).
    """
    final = []
    buf = ""

    def flush_buf():
        nonlocal buf
        if buf and len(buf.split()) >= MIN_WORDS_SEGMENT:
            final.append(buf.strip())
        buf = ""

    for seg in segments:
        seg = seg.strip()
        if not seg:
            continue

        # try add to buffer
        if buf:
            tmp = buf + " " + seg
            if len(tmp.split()) <= HARD_MAX_WORDS:
                buf = tmp
            else:
                flush_buf()
                buf = seg
        else:
            buf = seg


        if len(buf.split()) > HARD_MAX_WORDS:
            sents = split_sentences_vi(buf)
            buf = ""
            tmp = ""
            for s in sents:
                if len((tmp + " " + s).split()) <= SOFT_MAX_WORDS:
                    tmp = (tmp + " " + s).strip()
                else:
                    if len(tmp.split()) >= MIN_WORDS_SEGMENT:
                        final.append(tmp)
                    tmp = s
            if tmp:
                if len(tmp.split()) >= MIN_WORDS_SEGMENT:
                    final.append(tmp)
                else:
                    # nếu phần dư ngắn quá, dồn vào đoạn trước nếu có
                    if final and len(final[-1].split()) <= HARD_MAX_WORDS:
                        final[-1] = (final[-1] + " " + tmp).strip()
                    else:
                        buf = tmp

        elif len(buf.split()) >= SOFT_MAX_WORDS:
            flush_buf()

    flush_buf()
    return final

In [ ]:
def process_nested_folders(root_folder_path):
    rows = []

    if not os.path.exists(root_folder_path):
        print("Không tìm thấy thư mục:", root_folder_path)
        return pd.DataFrame(columns=["company","year","source_file","text"])

    for folder_name in sorted(os.listdir(root_folder_path)):
        sub_folder = os.path.join(root_folder_path, folder_name)
        if not os.path.isdir(sub_folder):
            continue

        company = folder_name.strip().upper()
        print(f"\n>> CÔNG TY: {company}")

        for fname in sorted(os.listdir(sub_folder)):
            if not fname.lower().endswith(".pdf"):
                continue
            path = os.path.join(sub_folder, fname)

            m = re.search(r'(\d{4})', fname)
            if not m:
                print(" - Bỏ (không có năm):", fname)
                continue
            year = int(m.group(1))

            print(f" - Đọc: {fname}")

            pages_data = read_pdf_text_ocr_per_page(path)
            if not pages_data:
                continue

            # lọc theo heading (scope), nhưng không lưu section/page
            relevant_text = extract_relevant_text_by_headings(pages_data)

            # clean mạnh
            cleaned_text = clean_text_and_remove_tables(relevant_text)
            if not cleaned_text or len(cleaned_text.split()) < 200:
                print("   -> Text quá ít sau clean, bỏ.")
                continue

            # segment + chunk
            segs = segment_by_paragraph(cleaned_text)
            chunks = chunk_segments(segs)

            # lưu
            added = 0
            for ch in chunks:
                if len(ch.split()) < MIN_WORDS_SEGMENT:
                    continue
                rows.append({
                    "company": company,
                    "year": year,
                    "source_file": fname,
                    "text": ch
                })
                added += 1

            print(f"   -> Thêm {added} segments")

    df = pd.DataFrame(rows)
    return df

In [ ]:
df_segments = process_nested_folders(GLOBAL_DATA_FOLDER)

if df_segments.empty:
    print("Không có dữ liệu sau xử lý.")
else:

    df_segments["word_count"] = df_segments["text"].apply(lambda x: len(str(x).split()))
    df_segments = df_segments[df_segments["word_count"] >= MIN_WORDS_SEGMENT].copy()

    df_segments["norm"] = df_segments["text"].apply(lambda x: unidecode(str(x).lower().strip()))
    # drop duplicates mạnh: theo company-year-norm
    before = len(df_segments)
    df_segments = df_segments.drop_duplicates(subset=["company","year","norm"]).copy()
    after = len(df_segments)

    print("\n--- CHẤT LƯỢNG SAU LỌC ---")
    print("Segments:", after, "(drop dup:", before - after, ")")
    print("Min words:", df_segments["word_count"].min(), " | Median:", int(df_segments["word_count"].median()), " | Mean:", round(df_segments["word_count"].mean(), 1))

    # Lưu segments_clean.csv
    df_out = df_segments.drop(columns=["norm","word_count"], errors="ignore")
    df_out.to_csv(OUT_SEGMENTS_CSV, index=False)
    print("Saved:", OUT_SEGMENTS_CSV)
    display(df_out.head(20))

    if EXPORT_DOCS_YEAR:
      docs_year = (
          df_out.groupby(["company","year"])["text"]
                .apply(lambda x: " ".join(x.astype(str)))
                .reset_index()
      )
      docs_year.to_csv(OUT_DOCS_YEAR_CSV, index=False)
      print("Saved:", OUT_DOCS_YEAR_CSV)
      display(docs_year.head(10))

#2. Chạy mô hình LDA

**TIỀN XỬ LÝ DỮ LIỆU**

In [ ]:
os.listdir("/content/drive/MyDrive")
path = "/content/drive/MyDrive/DT/segments_clean.csv"
df = pd.read_csv(path)

df.head()

**“LDA-ready preprocessing” cho tiếng Việt**

In [ ]:
df = df.rename(columns={
    "company": "Company",
    "year": "Year",
    "source_file": "SourceFile",
    "text": "Text"
})


df = df[["Company", "Year", "SourceFile", "Text"]].copy()


df["doc_id"] = df.index.astype(int)


df["Text"] = df["Text"].astype(str)
df["n_chars"] = df["Text"].str.len()
df["n_chars"].describe()

In [ ]:
import re
from unidecode import unidecode

def basic_clean(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"http\S+|www\.\S+", " ", s)
    s = re.sub(r"\S+@\S+", " ", s)
    s = re.sub(r"\+?\d[\d\-\s]{7,}\d", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["Text_clean"] = df["Text"].apply(basic_clean)
df = df[df["Text_clean"].str.len() >= 80].copy()
# Chuẩn hóa số để giảm nhiễu
df["Text_clean"] = df["Text_clean"].str.replace(r"\d+([.,]\d+)?%?", " <NUM> ", regex=True)
df["Text_clean"] = df["Text_clean"].str.replace(r"\s+", " ", regex=True).str.strip()

# dùng bản không dấu để match regex ổn định
df["Text_clean_ud"] = df["Text_clean"].apply(unidecode)

TOC_RE = re.compile(r"(muc luc|chu thich|viet tat)|(\.{5,})", re.IGNORECASE)
df = df[~df["Text_clean_ud"].str.contains(TOC_RE, regex=True, na=False)].copy()


MENU_RE = re.compile(r"\b0[1-9]\s+(thong|dau|tong|ket|phan|quan|bao|co)\b", re.IGNORECASE)
df = df[~df["Text_clean_ud"].str.contains(MENU_RE, regex=True, na=False)]

FIN_RE = re.compile(
    r"(bao cao tai chinh|nam tai chinh ket thuc|bao cao bo phan|thuyet minh|bang can doi|luu chuyen tien te|kiem toan|mau so|b0\d)",
    re.IGNORECASE
)
df = df[~df["Text_clean_ud"].str.contains(FIN_RE, regex=True, na=False)].copy()

# 1) regex điện thoại / fax
phone_re = re.compile(r"(\+?\d[\d\-\s]{7,}\d)|(\(\d{2,4}\))")

# 2) regex địa chỉ / liên hệ
contact_re_ud = re.compile(r"(dien thoai|fax|email|website|tru so|dia chi|lien he)", re.IGNORECASE)
mask_noise = (
    df["Text_clean_ud"].str.contains(phone_re, regex=True, na=False) |
    df["Text_clean_ud"].str.contains(contact_re_ud, regex=True, na=False)
)

print("Số segment rác bị loại:", mask_noise.sum())

# 4) loại bỏ
df = df[~mask_noise].copy()
df["word_count2"] = df["Text_clean"].str.split().apply(len)
df = df[(df["word_count2"] >= 80) & (df["word_count2"] <= 300)].copy()

In [ ]:
print("Remaining segments:", len(df))
print(df["Text_clean"].sample(5).tolist())

In [ ]:
df.groupby(["Year"]).size().sort_index()
# nếu có Company:
df.groupby(["Company","Year"]).size().sort_values(ascending=False).head(30)

In [ ]:
!pip -q install underthesea

from underthesea import word_tokenize

def vn_tokenize(s: str):
    # word_tokenize trả về string có dấu cách giữa token
    t = word_tokenize(s, format="text")
    return t.split()

df["tokens_raw"] = df["Text_clean"].apply(vn_tokenize)
df["tokens_raw"].head()

In [ ]:
import re
from unidecode import unidecode

# STOPWORD
STOP_BASIC = set("""
và hoặc nhưng là của cho trong với về từ các những được đã đang sẽ này đó một một_số
tại theo khi để đến nhằm hơn lại như vì do rất ông bà anh chị
""".split())

STOP_DOMAIN = {
    "tập_đoàn","công_ty","báo_cáo","thường_niên","năm","quý",
    "việt_nam","hà_nội","vnđ","vnd","tỷ","triệu", "bks", "ban",
    "hội_đồng_quản_trị","hđqt","ban_lãnh_đạo","ban_điều_hành","ban_giám_đốc","ban_kiểm_soát",
    "cổ_đông"
}


STOP_EXTRA = {
    "<_num","_num","<num>","num",
    "tổng_giám_đốc",
    "ủy_viên", "nghị_quyết", "quyết_định", "biên_bản", "thư_ký",
    "ifrs","chuẩn_mực","kế_toán","thuyết_minh","ghi_nhận",
    "phải_thu","phải_trả","khấu_hao","dự_phòng","bảng_cân_đối","bctc",
}

STOP_EXTRA |= {"chủ_tịch","thành_viên","kiểm_soát","kinh_nghiệm","vị_trí"}

STOP_GOV_PHRASES = {
    "ủy_viên_hđqt","ủy_viên_độc_lập","thành_viên_hđqt",
    "chủ_tịch_hđqt","phó_chủ_tịch_hđqt","phó_tổng_giám_đốc"
}

# Stop theo tên công ty
company_stop = set(df["Company"].astype(str).str.lower().str.strip().unique().tolist())
company_stop = {w for w in company_stop if w and w != "nan" and len(w) >= 2}

stopwords = STOP_BASIC | STOP_DOMAIN | STOP_EXTRA | STOP_GOV_PHRASES | company_stop

# Mở rộng biến thể để khớp token dạng "_" và không dấu
stopwords |= {unidecode(w) for w in stopwords}
stopwords |= {w.replace("_", " ") for w in stopwords}
stopwords |= {w.replace(" ", "_") for w in stopwords}

_num_or_has_digit = re.compile(r"\d")
_only_punct = re.compile(r"^[\W_]+$")

# Prefix để bắt đúng loại token như "ủy_viên_hđqt", "nghị_quyết_..."
GOV_PREFIX = (
    "ủy_viên_", "thành_viên_", "phó_",
    "chủ_tịch_", "giám_đốc_", "tổng_giám_đốc_",
    "ban_", "nghị_quyết_", "quyết_định_"
)


def filter_tokens(tokens):
    out = []
    for w in tokens:
        w = str(w).lower().strip()

        if len(w) < 2:
            continue
        if w in ("<num>", "num", "<_num", "_num"):
            continue
        if _num_or_has_digit.search(w):
            continue
        if _only_punct.match(w):
            continue
        if w.startswith(GOV_PREFIX):
            continue
        if w in stopwords:
            continue

        out.append(w)
    return out

df["tokens"] = df["tokens_raw"].apply(filter_tokens)
df["tok_len"] = df["tokens"].apply(len)


df = df[df["tok_len"] >= 12].copy().reset_index(drop=True)


df["Text_tok"]   = df["tokens"].apply(lambda x: " ".join(x))
df["Text_clean"] = df["tokens"].apply(lambda x: " ".join(x).replace("_", " "))

print("Segments after filter:", len(df), "| tok_len:", df["tok_len"].describe()[["min","50%","max"]].to_dict())


Segments after filter: 2622 | tok_len: {'min': 21.0, '50%': 83.0, 'max': 185.0}


In [ ]:

df["Text_tok"] = df["tokens"].apply(lambda xs: " ".join(xs))
out_path = "/content/drive/MyDrive/DT/segments_final.csv"

df_out = df[[
    "doc_id",
    "Company",
    "Year",
    "SourceFile",
    "Text_clean",
    "Text_tok"
]].copy()

df_out.to_csv(out_path, index=False, encoding="utf-8-sig")
print("Saved", out_path)


**Dictionary & Corpus (gensim-ready)**

In [ ]:
!pip -q install gensim
from gensim import corpora
texts = df["tokens"].tolist()

In [ ]:
dictionary = corpora.Dictionary(texts)
print("Original vocab size:", len(dictionary))

In [ ]:
dictionary.filter_extremes(
    no_below=10,      # xuất hiện ít nhất trong 10 documents
    no_above=0.5      # xuất hiện không quá 50% documents
)

dictionary.compactify()
print("Filtered vocab size:", len(dictionary))

In [ ]:
corpus = [dictionary.doc2bow(text) for text in texts]

# kiểm tra 1 doc
corpus[0][:10], [dictionary[id] for id, _ in corpus[0][:10]]

([(0, 1),
  (1, 1),
  (2, 1),
  (3, 1),
  (4, 1),
  (5, 1),
  (6, 1),
  (7, 1),
  (8, 1),
  (9, 1)],
 ['ai',
  'bản_thân',
  'chính',
  'chưa',
  'có',
  'công_nghiệp',
  'công_nghệ',
  'cần',
  'giai_đoạn',
  'hình_thành'])

In [ ]:
# số token trung bình sau BoW
import numpy as np

doc_lengths = [sum(cnt for _, cnt in doc) for doc in corpus]
np.percentile(doc_lengths, [10, 25, 50, 75, 90])

array([53., 66., 77., 86., 94.])

**CHỌN SỐ TOPIC**

In [ ]:
from gensim.models import LdaModel
from gensim.models.coherencemodel import CoherenceModel

In [ ]:
def compute_coherence_values(dictionary, corpus, texts, k_list):
    results = []
    for k in k_list:
        lda = LdaModel(
            corpus=corpus,
            id2word=dictionary,
            num_topics=k,
            random_state=42,
            passes=10,
            alpha="auto",
            eta="auto"
        )
        cm = CoherenceModel(
            model=lda,
            texts=texts,
            dictionary=dictionary,
            coherence="c_v"
        )
        coherence = cm.get_coherence()
        results.append((k, coherence))
        print(f"K={k}, coherence={coherence:.4f}")
    return results

In [ ]:
k_list = [5, 7, 9, 11, 13, 15]
coherence_results = compute_coherence_values(
    dictionary=dictionary,
    corpus=corpus,
    texts=texts,
    k_list=k_list
)

coherence_results

In [ ]:
import matplotlib.pyplot as plt

ks = [x[0] for x in coherence_results]
scores = [x[1] for x in coherence_results]

plt.figure(figsize=(7,4))
plt.plot(ks, scores, marker="o")
plt.xlabel("Number of Topics (K)")
plt.ylabel("Coherence Score (c_v)")
plt.title("Topic Coherence vs Number of Topics")
plt.grid(True)
plt.show()


Số lượng chủ đề được xác định bằng cách sử dụng c_v coherence score. Như thể hiện trong hình , coherence tăng đáng kể cho đến K = 8 và đạt mức tối đa tại K = 11. Vượt quá điểm này, oherence giảm, cho thấy sự phân mảnh chủ đề.
Do đó, K = 11 được chọn là số lượng chủ đề tối ưu.

In [ ]:
K = 11

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=K,
    random_state=42,
    passes=20,
    alpha="auto",
    eta="auto"
)

In [ ]:
topics = lda_model.show_topics(num_topics=K, num_words=10, formatted=False)

for topic_id, words in topics:
    print(f"\nTopic {topic_id}:")
    print(", ".join([w for w, _ in words]))

**Gán topic cho từng section**

In [ ]:
def get_dominant_topic(model, corpus):
    dominant_topics = []
    topic_probs = []

    for doc in corpus:
        topic_dist = model.get_document_topics(doc)
        if len(topic_dist) == 0:
            dominant_topics.append(None)
            topic_probs.append(0)
        else:
            top_topic, prob = max(topic_dist, key=lambda x: x[1])
            dominant_topics.append(top_topic)
            topic_probs.append(prob)

    return dominant_topics, topic_probs

In [ ]:
df["dominant_topic"], df["topic_prob"] = get_dominant_topic(lda_model, corpus)

df[["doc_id","Company","Year","SourceFile","dominant_topic","topic_prob"]].head()

In [ ]:
df["dominant_topic"].value_counts().sort_index()

In [ ]:
output_cols = [
    "doc_id","Company","Year","SourceFile",
    "dominant_topic","topic_prob"
]

df[output_cols].to_csv(
    "lda_topics.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
import matplotlib.pyplot as plt

def plot_topic_words(lda_model, topic_id, topn=10):
    words = lda_model.show_topic(topic_id, topn=topn)
    terms = [w for w, _ in words]
    weights = [p for _, p in words]

    plt.figure(figsize=(6,3))
    plt.barh(terms[::-1], weights[::-1])
    plt.title(f"Topic {topic_id} - Top {topn} words")
    plt.xlabel("Probability")
    plt.tight_layout()
    plt.show()

# ví dụ: vẽ 3 topic tiêu biểu
for t in [1, 6, 10]:
    plot_topic_words(lda_model, t)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# LDA - Topic share by Year

# 0) Safety checks (tránh lệch index giữa df và corpus)
assert len(df) == len(corpus), f"Mismatch: len(df)={len(df)} vs len(corpus)={len(corpus)}"
K = lda_model.num_topics  # hoặc giữ K bạn đã set = 11

# 1) Lấy full topic distribution cho mỗi doc (tổng = 1)
doc_topic_mat = np.zeros((len(corpus), K), dtype=float)

for i, bow in enumerate(corpus):
    # minimum_probability=0.0 để đảm bảo đủ K topics (topic nào không hiện sẽ là 0)
    for tid, prob in lda_model.get_document_topics(bow, minimum_probability=0.0):
        doc_topic_mat[i, tid] = prob

topic_cols = [f"topic_{t}" for t in range(K)]
df_doc_topics = pd.DataFrame(doc_topic_mat, columns=topic_cols)

# Gắn Year / doc_id theo đúng thứ tự df hiện tại
df_doc_topics["Year"] = df["Year"].astype(int).to_numpy()
df_doc_topics["doc_id"] = df["doc_id"].to_numpy()

# 2) Aggregate theo năm: mean(topic_probability) -> chính là "share" theo năm
topic_year = (
    df_doc_topics
    .groupby("Year")[topic_cols]
    .mean()
    .sort_index()
)

# normalize lại để tránh sai số số học rất nhỏ
topic_year = topic_year.div(topic_year.sum(axis=1), axis=0)

display(topic_year.head())

# 3) Tạo nhãn topic dễ nhìn (top words)
topic_names = {
    t: ", ".join([w for w, _ in lda_model.show_topic(t, topn=5)])
    for t in range(K)
}
rename_map = {f"topic_{t}": f"T{t}: {topic_names[t]}" for t in range(K)}
topic_year_named = topic_year.rename(columns=rename_map)

# 4) Plot 1: Stacked bar (toàn bộ topics theo năm)
ax = topic_year_named.plot(kind="bar", stacked=True, figsize=(11, 5))
ax.set_title("LDA - Topic Share by Year (mean topic probability)")
ax.set_xlabel("Year")
ax.set_ylabel("Share")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

# 5) Plot 2: Top-N topics mỗi năm (giống logic bạn làm cho BERTopic)
TOPN = 5

topic_year_long = (
    topic_year
    .reset_index()
    .melt(id_vars="Year", var_name="topic", value_name="share")
)
topic_year_long["topic_id"] = topic_year_long["topic"].str.replace("topic_", "", regex=False).astype(int)
topic_year_long["topic_name"] = topic_year_long["topic_id"].map(topic_names)

top_topics_year_lda = (
    topic_year_long
    .sort_values(["Year", "share"], ascending=[True, False])
    .groupby("Year")
    .head(TOPN)
    .copy()
)

display(top_topics_year_lda)

for y in sorted(top_topics_year_lda["Year"].unique()):
    tmp = top_topics_year_lda[top_topics_year_lda["Year"] == y].sort_values("share", ascending=True)
    plt.figure(figsize=(7, 3.2))
    plt.barh(tmp["topic_name"], tmp["share"])
    plt.title(f"LDA - Top {TOPN} Topics (Share) - Year {y}")
    plt.xlabel("Share")
    plt.tight_layout()
    plt.show()


#3. Chạy mô hình BERTopic


In [ ]:
import pandas as pd, re
from unidecode import unidecode
!pip -q install bertopic sentence-transformers umap-learn hdbscan
df = pd.read_csv("/content/drive/MyDrive/DT/segments_final.csv")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 10.4 MB/s eta 0:00:00


In [ ]:
# QUALITY GATE để giảm outlier (-1)
df["Text_tok"] = df["Text_tok"].astype(str).fillna("").str.strip()
df["tok_len"] = df["Text_tok"].str.split().str.len()

# cắt đoạn quá ngắn/quá dài
df = df[df["tok_len"].between(12, 220)].copy()

# dedup mềm theo Company-Year-text (giảm nhiễu lặp từ header/footer)
df["norm"] = df["Text_tok"].map(lambda s: unidecode(s.lower()).strip())
df = df.drop_duplicates(subset=["Company", "Year", "norm"]).copy()

df = df.reset_index(drop=True)
docs_tok2 = df["Text_tok"].tolist()
print("Segments after filter:", len(df), "| tok_len:", df["tok_len"].describe()[["min","50%","max"]].to_dict())


Segments after filter: 2622 | tok_len: {'min': 21.0, '50%': 83.0, 'max': 185.0}


In [ ]:
from sentence_transformers import SentenceTransformer
docs_clean = df["Text_clean"].astype(str).tolist()
embedding_model_name = "paraphrase-multilingual-mpnet-base-v2"
embedder = SentenceTransformer(embedding_model_name)

embeddings = embedder.encode(
    docs_clean,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)
print("Embeddings shape:", embeddings.shape)

In [ ]:
from bertopic import BERTopic
import umap, hdbscan
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import KeyBERTInspired

# 1) Làm sạch token nhiễu trước khi vectorize
docs_tok2_clean = [d.replace("<_num", " ").replace("_num", " ") for d in docs_tok2]

vi_stop_list = list(stopwords)

vectorizer_model = CountVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9,
    stop_words=vi_stop_list,
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None
)

umap_model = umap.UMAP(
    n_neighbors=35,
    n_components=8,
    min_dist=0.05,
    metric="cosine",
    random_state=42
)

hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=20,
    min_samples=2,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
    representation_model=KeyBERTInspired(),
    calculate_probabilities=True,
    verbose=True,
    embedding_model=embedder
)


topics, probs = topic_model.fit_transform(docs_tok2_clean, embeddings=embeddings)
topic_model.get_topic_info()

In [ ]:
import numpy as np

topics_arr = np.array(topics, dtype=int)
topics_fix = topics_arr.copy()

THRESH = 0.2

for i, t in enumerate(topics_arr):
    if t == -1 and probs[i] is not None:
        pmax = float(np.max(probs[i]))
        if pmax >= THRESH:
            topics_fix[i] = int(np.argmax(probs[i]))

print("Outlier before:", np.mean(topics_arr == -1))
print("Outlier after :", np.mean(topics_fix == -1))

In [ ]:
# Apply topics_fix vào model để các bước sau dùng đúng nhãn đã fix =

# 1) Ghi đè topics bằng topics_fix
topics = topics_fix.tolist()

print("Outlier rate (after your fix):", np.mean(np.array(topics) == -1))

# 2) Update model theo topics mới

topic_model.update_topics(docs_tok2_clean, topics=topics)




In [ ]:
# Reduce topics
try:
    topic_model.reduce_topics(docs_tok2_clean, nr_topics=13)
except TypeError:
    # Fallback cho version BERTopic cũ cần truyền topics/probs
    topic_model.reduce_topics(docs_tok2_clean, topics, probs, nr_topics=13)

# Transform lại để lấy topics/probs sau khi reduce
topics, probs = topic_model.transform(docs_tok2_clean, embeddings=embeddings)

print("N topics (after reduce):", len(set(topics)) - (1 if -1 in topics else 0))

# df_res dùng cho thống kê theo năm
df_res = df[["doc_id","Company","Year","SourceFile"]].copy()
df_res["topic"] = topics


In [ ]:
# Topic info sau reduce
topic_info = topic_model.get_topic_info()
display(topic_info)


In [ ]:
# Barchart: loại -1
topic_info = topic_model.get_topic_info()
top_topics = topic_info.query("Topic != -1").head(14)["Topic"].tolist()

fig_barchart = topic_model.visualize_barchart(
    topics=top_topics,
    top_n_topics=len(top_topics),
    n_words=12
)
fig_barchart



In [ ]:
# Hierarchy
fig_hierarchy = topic_model.visualize_hierarchy()
fig_hierarchy


In [ ]:
# Topics scatter map
fig_topics = topic_model.visualize_topics()
fig_topics


In [ ]:
# Heatmap
fig_heatmap = topic_model.visualize_heatmap()
fig_heatmap


In [ ]:

MANUAL_TOPIC_MAP = {
    -1: "Khác",
    0: "Quản trị chi phí, tài sản và nghĩa vụ hợp đồng",
    1: "Hệ sinh thái kinh doanh bất động sản–bán lẻ–du lịch/giải trí",
    2: "Đổi mới công nghệ & chiến lược xe điện (Vinfast)",
    3: "Quản trị môi trường và hiệu quả năng lượng",
    4: "Giáo dục và phát triển nguồn nhân lực (VinUni/VinSchool)",
    5: "Quản trị rủi ro và tuân thủ quy định",
    6: "Quản trị doanh nghiệp và mô hình vận hành",
    7: "Bối cảnh vĩ mô và triển vọng thị trường",
    8: "Dịch vụ y tế và chất lượng chăm sóc sức khỏe (Vinmec)",
    9: "Quản trị học thuật và nhân sự lãnh đạo trong cơ sở giáo dục",
    10: "Trách nhiệm xã hội và hoạt động thiện nguyện",
    11: "Văn hóa doanh nghiệp và gắn kết nhân viên",
}


In [ ]:
timestamps = df_res["Year"].tolist()
topics_over_time = topic_model.topics_over_time(
    docs_tok2_clean, timestamps, topics=topics, nr_bins=None
)

# bỏ -1 khỏi bảng/plot
tot_no_out = topics_over_time[topics_over_time.Topic != -1].copy()
display(tot_no_out.head(20))

fig_tot = topic_model.visualize_topics_over_time(tot_no_out)
fig_tot


#4. Tổng quan kết quả

In [ ]:
df_plot = df_res.copy()
df_plot["Year"] = df_plot["Year"].astype(int)

# bỏ outlier và topic rác
df_plot = df_plot[df_plot["topic"] != -1].copy()


# gán tên
df_plot["topic_name"] = df_plot["topic"].map(MANUAL_TOPIC_MAP).fillna("Khác / Chưa đặt tên")

df_plot["topic"] = df_plot["topic_name"]

# shares theo năm
shares = df_plot.groupby(["Year", "topic"]).size().reset_index(name="count")
shares["share"] = shares.groupby("Year")["count"].transform(lambda s: s / s.sum())
shares["topic_name"] = shares["topic"]
TOPN = 5
top_topics_year = (shares.sort_values(["Year","share"], ascending=[True, False])
                        .groupby("Year")
                        .head(TOPN))

display(top_topics_year)

In [ ]:
# Bar chart TOPN topics per year
import matplotlib.pyplot as plt

plot_df = top_topics_year.copy()
plot_df["label"] = plot_df["topic_name"].fillna(plot_df["topic"].astype(str))

years_sorted = sorted(plot_df["Year"].unique())
for y in years_sorted:
    tmp = plot_df[plot_df["Year"] == y].sort_values("share", ascending=True)
    plt.figure()
    plt.barh(tmp["label"], tmp["share"])
    plt.title(f"Top {TOPN} Topics - Year {y}")
    plt.xlabel("Share")
    plt.tight_layout()
    plt.show()


In [ ]:
def plot_topic_share_stacked(shares: pd.DataFrame, top_k: int = 8, other_label: str = "Other",
                             figsize=(10, 5)):

    req = {"Year","topic","share","topic_name"}
    missing = req - set(shares.columns)
    if missing:
        raise ValueError(f"shares thiếu cột: {missing}")

    # 1) chọn top_k topic theo mean share toàn giai đoạn
    topic_mean = (shares.groupby(["topic","topic_name"])["share"].mean()
                        .reset_index()
                        .sort_values("share", ascending=False))
    top_topics = topic_mean.head(top_k)["topic"].tolist()

    # 2) gom Other
    s = shares.copy()
    s["topic_plot"] = np.where(s["topic"].isin(top_topics), s["topic_name"], other_label)

    s_plot = (s.groupby(["Year","topic_plot"])["share"].sum()
                .reset_index())

    # 3) pivot để vẽ stacked bar
    pivot = (s_plot.pivot(index="Year", columns="topic_plot", values="share")
                  .fillna(0.0))

    # sort cột theo tổng share (để legend gọn)
    col_order = pivot.sum(axis=0).sort_values(ascending=False).index
    pivot = pivot[col_order]

    ax = pivot.plot(kind="bar", stacked=True, figsize=figsize, width=0.85)
    ax.set_title(f"Topic Share by Year (Top {top_k} + {other_label})")
    ax.set_xlabel("Year")
    ax.set_ylabel("Share")
    ax.set_ylim(0, 1)

    # legend ra ngoài cho dễ nhìn
    ax.legend(title="Topic", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
    plt.tight_layout()
    plt.show()

plot_topic_share_stacked(shares, top_k=13)


In [ ]:
def momentum_ranking(shares: pd.DataFrame) -> pd.DataFrame:

    rows = []
    for (topic, name), g in shares.groupby(["topic","topic_name"], sort=False):
        gg = g.sort_values("Year")
        years = gg["Year"].to_numpy(dtype=float)
        y = gg["share"].to_numpy(dtype=float)

        # bỏ topic có ít năm quá (ví dụ chỉ xuất hiện 1-2 năm)
        if len(years) < 3:
            continue

        share_first = float(y[0])
        share_last  = float(y[-1])
        delta = share_last - share_first

        # slope bằng regression tuyến tính y = a*year + b
        a, b = np.polyfit(years, y, 1)  # a = slope per year

        # R^2
        y_hat = a*years + b
        ss_res = float(((y - y_hat)**2).sum())
        ss_tot = float(((y - y.mean())**2).sum())
        r2 = 1.0 - ss_res/ss_tot if ss_tot > 0 else np.nan

        rows.append({
            "topic": topic,
            "topic_name": name,
            "first_year": int(years[0]),
            "last_year": int(years[-1]),
            "share_first": share_first,
            "share_last": share_last,
            "delta": delta,
            "slope_per_year": float(a),
            "r2": r2,
            "n_years": int(len(years))
        })

    out = pd.DataFrame(rows)
    out = out.sort_values(["slope_per_year","delta"], ascending=[False, False]).reset_index(drop=True)
    return out

rank = momentum_ranking(shares)

# Top rising / falling
N = 5
top_rising  = rank.head(N).copy()
top_falling = rank.tail(N).sort_values("slope_per_year").copy()

print("Top rising:")
display(top_rising)
print("Top falling:")
display(top_falling)


In [ ]:
def plot_topic_lines(shares: pd.DataFrame, topics_to_plot, title: str, figsize=(9,4.8)):
    s = shares[shares["topic"].isin(topics_to_plot)].copy()
    pivot = (s.pivot_table(index="Year", columns="topic_name", values="share", aggfunc="sum")
               .fillna(0.0)
               .sort_index())

    ax = pivot.plot(kind="line", marker="o", figsize=figsize)
    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.set_ylabel("Share")
    ax.set_ylim(0, max(0.01, float(pivot.values.max())*1.15))
    ax.legend(title="Topic", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)

    plt.tight_layout()
    plt.show()

# chọn topic id theo ranking ở trên
rising_topics = top_rising["topic"].tolist()
falling_topics = top_falling["topic"].tolist()

plot_topic_lines(shares, rising_topics,  title=f"Top {N} Rising Topics (by slope)")
plot_topic_lines(shares, falling_topics, title=f"Top {N} Falling Topics (by slope)")


In [ ]:
import pandas as pd
import numpy as np

def build_year_theme_table_from_bertopic(
    df_res: pd.DataFrame,
    year_col: str = "Year",
    topic_col: str = "topic",
    manual_map: dict = None,
    drop_topics: set = None,
    drop_outlier: bool = True,
    min_years: int = 1
) -> pd.DataFrame:

    if manual_map is None:
        raise ValueError("Bạn cần truyền manual_map (dict topic_id -> tên chủ đề).")

    s = df_res[[year_col, topic_col]].copy()
    s[year_col] = s[year_col].astype(int)

    if drop_outlier:
        s = s[s[topic_col] != -1].copy()

    if drop_topics:
        s = s[~s[topic_col].isin(drop_topics)].copy()

    # gán tên thủ công
    s["Theme"] = s[topic_col].map(manual_map).fillna("Khác / Chưa đặt tên")

    # count và share theo năm
    agg = (s.groupby([year_col, "Theme"])
             .size()
             .reset_index(name="count"))

    agg["share"] = agg.groupby(year_col)["count"].transform(lambda x: x / x.sum())

    # lọc chủ đề xuất hiện ít nhất min_years
    if min_years > 1:
        n_year = agg.groupby("Theme")[year_col].nunique()
        keep = n_year[n_year >= min_years].index
        agg = agg[agg["Theme"].isin(keep)].copy()

    # pivot ra bảng Year x Theme
    table = (agg.pivot(index=year_col, columns="Theme", values="share")
               .fillna(0.0)
               .sort_index())

    # sắp cột theo mean share giảm dần
    col_order = table.mean(axis=0).sort_values(ascending=False).index
    table = table[col_order]

    return table

year_theme_table = build_year_theme_table_from_bertopic(
    df_res=df_res,
    year_col="Year",
    topic_col="topic",
    manual_map=MANUAL_TOPIC_MAP,
    drop_topics=None,
    drop_outlier=True
)

# làm đẹp: làm tròn 2 chữ số
display(year_theme_table.round(2))


In [ ]:
TOPN = 5
topn_each_year = (year_theme_table
                  .apply(lambda row: row.sort_values(ascending=False).head(TOPN), axis=1))

display(topn_each_year.round(2))


In [ ]:
topn_long = (year_theme_table
             .stack()
             .reset_index()
             .rename(columns={0:"share"})
             .sort_values(["Year","share"], ascending=[True,False])
             .groupby("Year")
             .head(TOPN))

display(topn_long.round(3))
